# einops-einsum — ex9: three-tensor Tucker-style contraction with intermediate-shape debug prints

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-einsum`. Running the final beacon cell reports progress against the `Einops: Deep Learning` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Deep Learning` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-einsum`** (exercise 9). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-einsum"
DD_SUBTOPIC = "Einops: Deep Learning"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.einsum — quick refresher

`einsum(*tensors, pattern)` performs sum-contraction over named indices:
1. **Elementwise** — `'i j, i j -> i j'` multiplies pointwise (no reduction).
2. **Matmul** — `'i k, k j -> i j'` contracts the shared `k` (sum-reduce).
3. **Batched** — `'b i k, b k j -> b i j'` carries `b` through, contracts `k`.
4. **Three operands** — `'i j, j k, k l -> i l'` chains two contractions; the optimizer picks pairing order.

**The two rules:**
- An index that appears on input AND output → preserved (broadcast-like).
- An index that appears on input but NOT on output → sum-contracted.

### Exercise 9 — three-tensor Tucker-style contraction with intermediate-shape debug prints

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Use a single 3-tensor einsum to project a core (P, Q, R) tensor onto three factor matrices U, V, W in one pattern, and compare its result against a hand-rolled 3-step pairwise contraction that prints each intermediate shape.
> Keywords: tucker, three-tensor, pairing-order, debug-print
> ```

**KCs targeted:** `einsum-three-operand-contraction`, `einsum-pairwise-vs-fused`, `einsum-debug-introspection`

Implement TWO functions:

1. `ex9_tucker_fused(core, U, V, W)` — one-shot 3-operand einsum.
   - `core`: `(P, Q, R)`.
   - `U`: `(I, P)`. `V`: `(J, Q)`. `W`: `(K, R)`.
   - Output: `(I, J, K)` where `out[i,j,k] = sum_p sum_q sum_r U[i,p] * V[j,q] * W[k,r] * core[p,q,r]`.
   - Use ONE `einops.einsum` call with all four operands and one pattern.

2. `ex9_tucker_pairwise(core, U, V, W)` — hand-rolled 3-step pairwise version, for comparison.
   - Step 1: contract `p` between `U` and `core` → shape `(I, Q, R)`. Print: `step1 shape=...`.
   - Step 2: contract `q` between `V` and step1 → shape `(I, J, R)`. Print: `step2 shape=...`.
   - Step 3: contract `r` between `W` and step2 → shape `(I, J, K)`. Print: `step3 shape=...`.
   - Each step uses a separate `einops.einsum` call.

The test cell calls both and asserts they produce the **same** tensor (up to floating-point tolerance). It also greps the debug log for the three step prints.

In [ ]:
def ex9_tucker_fused(core: Tensor, U: Tensor, V: Tensor, W: Tensor) -> Tensor:
    """One-shot 3-mode Tucker reconstruction via a single 4-operand einsum."""
    raise NotImplementedError()


def ex9_tucker_pairwise(core: Tensor, U: Tensor, V: Tensor, W: Tensor) -> Tensor:
    """Same Tucker reconstruction but as 3 pairwise einsums.
    Must print `step1 shape=...`, `step2 shape=...`, `step3 shape=...`."""
    raise NotImplementedError()


def _test_ex9():
    import io, contextlib

    P, Q, R = 3, 4, 5
    I, J, K = 6, 7, 2
    core = t.randn(P, Q, R)
    U = t.randn(I, P)
    V = t.randn(J, Q)
    W = t.randn(K, R)

    fused = ex9_tucker_fused(core, U, V, W)
    assert fused.shape == (I, J, K), f'fused: expected ({I},{J},{K}), got {fused.shape}'

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        pairwise = ex9_tucker_pairwise(core, U, V, W)
    log = buf.getvalue()
    print(log, end='')

    assert pairwise.shape == (I, J, K), f'pairwise: expected ({I},{J},{K}), got {pairwise.shape}'
    assert t.allclose(fused, pairwise, atol=1e-4), (
        'fused vs pairwise disagree — same math should give the same answer'
    )

    # Independent ground truth via explicit nested matmuls.
    # m1[i,q,r] = sum_p U[i,p] * core[p,q,r]  ==  (U @ core.reshape(P, Q*R)).reshape(I,Q,R)
    m1 = (U @ core.reshape(P, Q * R)).reshape(I, Q, R)
    m2 = (V @ m1.permute(1, 0, 2).reshape(Q, I * R)).reshape(J, I, R).permute(1, 0, 2)
    expected = (W @ m2.permute(2, 0, 1).reshape(R, I * J)).reshape(K, I, J).permute(1, 2, 0)
    assert t.allclose(fused, expected, atol=1e-4), 'fused disagrees with reference'

    # Debug-print contract.
    for needle in ['step1', 'step2', 'step3']:
        assert needle in log, f'missing `{needle}` debug print:\n{log}'
    assert f'({I}, {Q}, {R})' in log, f'step1 should report ({I},{Q},{R}):\n{log}'
    assert f'({I}, {J}, {R})' in log, f'step2 should report ({I},{J},{R}):\n{log}'
    assert f'({I}, {J}, {K})' in log, f'step3 should report ({I},{J},{K}):\n{log}'

    # Visualize the (I, J) slice at k=0 of the fused result as a heatmap.
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(fused[..., 0].numpy(), cmap='coolwarm')
    ax.set_title(f'Tucker reconstruction[:,:,0]  shape={tuple(fused.shape)}')
    ax.set_xlabel('j'); ax.set_ylabel('i')
    plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex9')
    print("ex9 ✓")

_test_ex9()

<details><summary>Solution</summary>

```python
def ex9_tucker_fused(core: Tensor, U: Tensor, V: Tensor, W: Tensor) -> Tensor:
    return einsum(core, U, V, W, 'p q r, i p, j q, k r -> i j k')


def ex9_tucker_pairwise(core: Tensor, U: Tensor, V: Tensor, W: Tensor) -> Tensor:
    step1 = einsum(U, core, 'i p, p q r -> i q r')
    print(f"step1 shape={tuple(step1.shape)}")
    step2 = einsum(V, step1, 'j q, i q r -> i j r')
    print(f"step2 shape={tuple(step2.shape)}")
    step3 = einsum(W, step2, 'k r, i j r -> i j k')
    print(f"step3 shape={tuple(step3.shape)}")
    return step3
```

**Tucker decomposition recap.** A Tucker reconstruction of a 3D tensor is a small "core" tensor `core ∈ ℝ^{P×Q×R}` projected back to full size by three factor matrices `U ∈ ℝ^{I×P}`, `V ∈ ℝ^{J×Q}`, `W ∈ ℝ^{K×R}`. The fused pattern `'p q r, i p, j q, k r -> i j k'` contracts `p`, `q`, `r` (all the small dims) and keeps `i`, `j`, `k` (the full dims).

**Pairwise vs fused is the same math.** The fused einsum is mathematically identical to the 3-step pairwise version — `einops.einsum` delegates to `torch.einsum`, which uses `opt_einsum` to pick a contraction order automatically. Writing it out as 3 steps lets you **see** the intermediate shapes, which is the whole point of the debug prints: at each step, exactly one small index is consumed and one full index is materialized. The intermediate `(I, Q, R)` is small if `I ≪ Q*R`, large if `I ≫ Q*R` — and that choice is what `opt_einsum` agonizes over for big tensors.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex9'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex9',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()